# MartiniSurf Colab: Protein + AutoMartini M3
### User-friendly workflow (no coding required)

This notebook lets you:
- install MartiniSurf
- optionally generate a linker from SMILES or uploaded structure
- build a protein-surface system
- visualize the final `immobilized_system.gro`

## 1) Install

In [ ]:
#@title Install MartiniSurf and Viewer
INSTALL_FROM_GITHUB = True #@param {type:"boolean"}
GITHUB_REPO = "https://github.com/jjimenezgar/MartiniSurf.git" #@param {type:"string"}

import subprocess, os

if INSTALL_FROM_GITHUB:
    subprocess.run("pip -q install --upgrade pip", shell=True, check=True)
    subprocess.run(f"git clone {GITHUB_REPO} /content/MartiniSurf", shell=True, check=False)
    subprocess.run("pip -q install -e /content/MartiniSurf", shell=True, check=True)
    subprocess.run("pip -q install py3Dmol", shell=True, check=True)
    print("MartiniSurf installed in /content/MartiniSurf")
else:
    print("Installation skipped")


## 2) Optional: Generate Linker (AutoMartini M3)

In [ ]:
#@title Optional linker generation (M3)
RUN_LINKER_GENERATION = False #@param {type:"boolean"}
INPUT_MODE = "SMILES" #@param ["SMILES", "Upload file"]
SMILES = "CCO" #@param {type:"string"}
MOLECULE_NAME = "LNK" #@param {type:"string"}
MOLECULE_CHARGE = 0 #@param {type:"integer"}
OUTPUT_DIR = "/content/generated_linker_m3" #@param {type:"string"}

from pathlib import Path
import subprocess

if RUN_LINKER_GENERATION:
    subprocess.run("rm -rf /content/Automartini_M3", shell=True, check=False)
    subprocess.run("git clone https://github.com/Martini-Force-Field-Initiative/Automartini_M3 /content/Automartini_M3", shell=True, check=True)

    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    if INPUT_MODE == "Upload file":
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded")
        fpath = "/content/" + next(iter(uploaded.keys()))
        cmds = [
            f"python AutoMartini.py --mol '{fpath}' --molname {MOLECULE_NAME} --charge {MOLECULE_CHARGE} --output '{OUTPUT_DIR}'",
            f"python AutoMartini.py --input '{fpath}' --molname {MOLECULE_NAME} --charge {MOLECULE_CHARGE} --output '{OUTPUT_DIR}'",
        ]
    else:
        cmds = [
            f"python AutoMartini.py --smi '{SMILES}' --molname {MOLECULE_NAME} --charge {MOLECULE_CHARGE} --output '{OUTPUT_DIR}'",
        ]

    ok = False
    for cmd in cmds:
        print("Trying:", cmd)
        r = subprocess.run(cmd, shell=True, cwd="/content/Automartini_M3")
        if r.returncode == 0:
            ok = True
            break

    if not ok:
        raise RuntimeError(
            "AutoMartini M3 command failed. Check /content/Automartini_M3/README.md for updated CLI flags."
        )

    print("Linker generation finished. Files:")
    subprocess.run(f"ls -lh '{OUTPUT_DIR}'", shell=True, check=False)
else:
    print("Skipped")


## 3) Build Protein System

In [ ]:
#@title Build system (Protein)
INPUT_PDB_MODE = "PDB ID" #@param ["PDB ID", "Upload file"]
PDB_ID = "1RJW" #@param {type:"string"}
MOLTYPE = "Protein" #@param {type:"string"}
OUTDIR = "/content/Simulation_Files" #@param {type:"string"}

USE_EXISTING_SURFACE = False #@param {type:"boolean"}
SURFACE_GRO = "/content/surface.gro" #@param {type:"string"}
LX = 20.0 #@param {type:"number"}
LY = 20.0 #@param {type:"number"}
SURFACE_BEAD = "C1" #@param {type:"string"}

USE_LINKER = False #@param {type:"boolean"}
LINKER_GRO = "/content/linker.gro" #@param {type:"string"}
INVERT_LINKER = False #@param {type:"boolean"}

ANCHOR_1 = "1 8 10 11" #@param {type:"string"}
ANCHOR_2 = "2 1025 1027 1028" #@param {type:"string"}
DIST = 10.0 #@param {type:"number"}

import subprocess, shlex

if INPUT_PDB_MODE == "Upload file":
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No PDB uploaded")
    pdb_input = "/content/" + next(iter(uploaded.keys()))
else:
    pdb_input = PDB_ID

cmd = ["martinisurf", "--pdb", pdb_input, "--moltype", MOLTYPE, "--outdir", OUTDIR]

if USE_EXISTING_SURFACE:
    cmd += ["--surface", SURFACE_GRO]
else:
    cmd += ["--lx", str(LX), "--ly", str(LY), "--surface-bead", SURFACE_BEAD]

if USE_LINKER:
    cmd += ["--linker", LINKER_GRO]
    if INVERT_LINKER:
        cmd += ["--invert-linker"]
    cmd += ["--linker-group"] + ANCHOR_1.split()
else:
    cmd += ["--anchor"] + ANCHOR_1.split()
    if ANCHOR_2.strip():
        cmd += ["--anchor"] + ANCHOR_2.split()
    cmd += ["--dist", str(DIST)]

print("Running:
", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)
print("Build completed")


## 4) Download Results

In [ ]:
#@title Download simulation folder as ZIP
SIM_DIR = "/content/Simulation_Files" #@param {type:"string"}
ZIP_NAME = "Simulation_Files" #@param {type:"string"}

import shutil
from google.colab import files

shutil.make_archive(ZIP_NAME, 'zip', SIM_DIR)
files.download(ZIP_NAME + '.zip')


## 5) Visualize Generated Structure

In [ ]:
#@title Viewer (py3Dmol)
SYSTEM_GRO = "/content/Simulation_Files/2_system/immobilized_system.gro" #@param {type:"string"}
SHOW_STICKS = False #@param {type:"boolean"}

import py3Dmol
from pathlib import Path

p = Path(SYSTEM_GRO)
if not p.exists():
    raise FileNotFoundError(f"File not found: {SYSTEM_GRO}")

model = p.read_text()
view = py3Dmol.view(width='100%', height=700)
view.addModel(model, 'gro')
if SHOW_STICKS:
    view.setStyle({'stick':{}})
else:
    view.setStyle({'sphere': {'radius': 1.4}})
view.zoomTo()
view.show()


---
Done. Re-run only the sections you need.